In [3]:
# =========================================================
# OTT 앱 리뷰 통합 수집기
# - Google Play + Apple App Store
# - Coupang Play / Netflix / TVING / Disney+
# - 기간 필터링 가능
# - CSV 저장
# =========================================================

# 설치 필요 패키지
# pip install google-play-scraper app-store-scraper pandas tqdm

import pandas as pd
from tqdm import tqdm
from datetime import datetime

# -----------------------------
# Google Play
# -----------------------------
from google_play_scraper import reviews, Sort

# -----------------------------
# Apple App Store
# -----------------------------
from app_store_web_scraper import AppStore


# =========================================================
# 수집 기간 설정
# =========================================================

START_DATE = datetime(2025, 4, 1)
END_DATE   = datetime(2026, 4, 30)


# =========================================================
# 앱 정보
# =========================================================

APPS = {
    "Coupang Play": {
        "google_play_id": "com.coupang.mobile.play",
        "appstore_id": 1558720674
    },
    "Netflix": {
        "google_play_id": "com.netflix.mediaclient",
        "appstore_id": 363590051
    },
    "TVING": {
        "google_play_id": "net.cj.cjhv.gs.tving",
        "appstore_id": 1481335069
    },
    "Disney+": {
        "google_play_id": "com.disney.disneyplus",
        "appstore_id": 1446075923
    }
}


# =========================================================
# Google Play 리뷰 수집
# =========================================================

def collect_google_reviews(app_name, app_id, count=5000):

    print(f"\n[Google Play] {app_name} 수집 시작")

    result, _ = reviews(
        app_id,
        lang='ko',
        country='kr',
        sort=Sort.NEWEST,
        count=count
    )

    rows = []

    for r in tqdm(result):

        review_date = r['at']

        if START_DATE <= review_date <= END_DATE:

            rows.append({
                "app": app_name,
                "store": "Google Play",
                "date": review_date,
                "rating": r['score'],
                "review": r['content'],
                "thumbs_up": r['thumbsUpCount'],
                "version": r.get('reviewCreatedVersion'),
            })

    return pd.DataFrame(rows)


# =========================================================
# App Store 리뷰 수집
# =========================================================

def collect_appstore_reviews(app_name, app_id):

    print(f"\n[App Store] {app_name} 수집 시작")

    app = AppStore(
        country="kr",
        app_name=app_name,
        app_id=app_id
    )

    app.review()

    rows = []

    for r in tqdm(app.reviews):

        review_date = r['date']

        if START_DATE <= review_date <= END_DATE:

            rows.append({
                "app": app_name,
                "store": "App Store",
                "date": review_date,
                "rating": r['rating'],
                "review": r['review'],
                "thumbs_up": None,
                "version": r.get('version')
            })

    return pd.DataFrame(rows)


# =========================================================
# 전체 수집
# =========================================================

all_reviews = []

for app_name, info in APPS.items():

    # -------------------------
    # Google Play
    # -------------------------
    try:
        gp_df = collect_google_reviews(
            app_name,
            info["google_play_id"],
            count=5000
        )

        all_reviews.append(gp_df)

    except Exception as e:
        print(f"[ERROR][Google Play] {app_name}: {e}")

    # -------------------------
    # App Store
    # -------------------------
    try:
        ios_df = collect_appstore_reviews(
            app_name,
            info["appstore_id"]
        )

        all_reviews.append(ios_df)

    except Exception as e:
        print(f"[ERROR][App Store] {app_name}: {e}")


# =========================================================
# 데이터 통합
# =========================================================

final_df = pd.concat(all_reviews, ignore_index=True)

# 중복 제거
final_df.drop_duplicates(
    subset=["app", "store", "date", "review"],
    inplace=True
)

# 날짜 정렬
final_df = final_df.sort_values("date", ascending=False)

# =========================================================
# CSV 저장
# =========================================================

filename = "ott_reviews_2025_2026.csv"

final_df.to_csv(
    filename,
    index=False,
    encoding="utf-8-sig"
)

print("\n====================================")
print("수집 완료")
print(f"총 리뷰 수: {len(final_df):,}")
print(f"저장 파일: {filename}")
print("====================================")


# =========================================================
# 간단 확인
# =========================================================

print(final_df.head())

ImportError: cannot import name 'AppStore' from 'app_store_web_scraper' (C:\Users\SAMSUNG\anaconda3\Lib\site-packages\app_store_web_scraper\__init__.py)

In [4]:
from google_play_scraper import reviews

result, _ = reviews(
    'com.netflix.mediaclient',
    lang='ko',
    country='kr',
    count=5
)

print(result[0])

{'reviewId': 'd714176a-1a58-4139-91bf-344aad623299', 'userName': 'Dasol Kang (피터슨s 블루스)', 'userImage': 'https://play-lh.googleusercontent.com/a-/ALV-UjWxn2lnMoc1NLZvofn3iYiOTECpYRRVDWYSlW3SWsKv_0x1TUCc', 'content': '자동으로 추가 계정 생긴거 좀 거슬림', 'score': 3, 'thumbsUpCount': 0, 'reviewCreatedVersion': '9.65.0 build 9 64092', 'at': datetime.datetime(2026, 5, 12, 20, 8, 19), 'replyContent': None, 'repliedAt': None, 'appVersion': '9.65.0 build 9 64092'}


In [5]:
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from google_play_scraper import reviews, Sort

START_DATE = datetime(2025, 4, 1)
END_DATE   = datetime(2026, 4, 30)

APPS = {
    "Coupang Play": "com.coupang.mobile.play",
    "Netflix": "com.netflix.mediaclient",
    "TVING": "net.cj.cjhv.gs.tving",
    "Disney+": "com.disney.disneyplus"
}

all_reviews = []

for app_name, app_id in APPS.items():

    print(f"\n수집 중: {app_name}")

    result, _ = reviews(
        app_id,
        lang='ko',
        country='kr',
        sort=Sort.NEWEST,
        count=5000
    )

    for r in tqdm(result):

        review_date = r['at']

        if START_DATE <= review_date <= END_DATE:

            all_reviews.append({
                "app": app_name,
                "date": review_date,
                "rating": r['score'],
                "review": r['content']
            })

df = pd.DataFrame(all_reviews)

df.to_csv(
    "ott_googleplay_reviews.csv",
    index=False,
    encoding="utf-8-sig"
)

print(df.head())
print(f"\n총 리뷰 수: {len(df):,}")


수집 중: Coupang Play


100%|██████████| 5000/5000 [00:00<00:00, 43059.49it/s]



수집 중: Netflix


100%|██████████| 5000/5000 [00:00<00:00, 533273.66it/s]



수집 중: TVING


100%|██████████| 5000/5000 [00:00<00:00, 424481.73it/s]



수집 중: Disney+


100%|██████████| 5000/5000 [00:00<00:00, 835985.01it/s]

            app                date  rating  \
0  Coupang Play 2026-04-29 16:33:00       5   
1  Coupang Play 2026-04-29 10:57:26       1   
2  Coupang Play 2026-04-28 20:32:55       3   
3  Coupang Play 2026-04-28 19:01:03       1   
4  Coupang Play 2026-04-28 13:42:55       1   

                                              review  
0                                                재밌다  
1                             돈 내고 보는데 광고까지 봐야 해요???  
2                       패스 사서 보는건데 광고가 있는 이유는 무엇인가요?  
3                                             로그인 안됨  
4  와우 회원도 중간 광고. 15초도 아니고 건너뛰기도 없고, 23, 30초짜리 넣어놓...  

총 리뷰 수: 5,713


In [7]:
import requests
import pandas as pd
from datetime import datetime
from tqdm import tqdm

START_DATE = datetime(2025, 4, 1)
END_DATE   = datetime(2026, 4, 30)

# 한국 App Store 앱 ID
APPS = {
    "Coupang Play": 1558720674,
    "Netflix": 363590051,
    "TVING": 1481335069,
    "Disney+": 1446075923
}

all_reviews = []

for app_name, app_id in APPS.items():

    print(f"\n수집 중: {app_name}")

    # 최대 10페이지 정도 반복
    for page in range(1, 11):

        url = f"https://itunes.apple.com/kr/rss/customerreviews/page={page}/id={app_id}/sortby=mostrecent/json"

        try:
            response = requests.get(url)

            if response.status_code != 200:
                continue

            data = response.json()

            entries = data['feed'].get('entry', [])

            # 첫 entry는 앱 정보라 제외
            entries = entries[1:]

            for entry in tqdm(entries):

                review_date = datetime.strptime(
                    entry['updated']['label'],
                    "%Y-%m-%dT%H:%M:%S%z"
                ).replace(tzinfo=None)

                if START_DATE <= review_date <= END_DATE:

                    all_reviews.append({
                        "app": app_name,
                        "store": "App Store",
                        "date": review_date,
                        "rating": int(entry['im:rating']['label']),
                        "review": entry['content']['label']
                    })

        except Exception as e:
            print(f"에러 발생: {e}")

df_ios = pd.DataFrame(all_reviews)

df_ios.to_csv(
    "ott_appstore_reviews.csv",
    index=False,
    encoding="utf-8-sig"
)

print(df_ios.head())
print(f"\n총 리뷰 수: {len(df_ios):,}")


수집 중: Coupang Play


0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]



수집 중: Netflix


100%|██████████| 49/49 [00:00<00:00, 14536.77it/s]



수집 중: TVING


0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]



수집 중: Disney+


100%|██████████| 49/49 [00:00<00:00, 18497.07it/s]

       app      store                date  rating  \
0  Netflix  App Store 2026-04-29 20:51:32       5   
1  Netflix  App Store 2026-04-29 18:08:18       1   
2  Netflix  App Store 2026-04-29 00:42:20       1   
3  Netflix  App Store 2026-04-26 07:47:17       5   
4  Netflix  App Store 2026-04-25 15:49:01       5   

                                              review  
0                                              기모찌야르  
1  아니 아이패드로 프로필 비번 설정하고 사용중인데 프로필누르고 비번 입력할때 하나 누...  
2  모든 나라에서 모든 작품을 볼수있도록 해주세요. 어떤나라에는 있고 한국에는없고 이런...  
3                                              너무좋아요  
4  17,000원 주고 매달 프리미엄 쓰고 있는데 추가회원을 또 매달 4,000원 내라...  

총 리뷰 수: 812
